# NorthStar Urban Mobility - SQL in R Analysis
## Part 1: SQL Querying within R using RSQLite

This notebook loads the NorthStar dataset into an SQLite database within R, performs SQL queries to investigate operational inefficiencies, and demonstrates query optimisation through indexing.

## Setup
Run this cell first to clone the data repository.

In [ ]:
import os
if not os.path.exists('northstar-coursework'):
    !git clone https://github.com/Erucard/northstar-coursework.git
os.chdir('/content/northstar-coursework/data/raw')
print('Ready:', sorted([f for f in os.listdir('.') if f.endswith('.csv')]))

In [ ]:
# Install and activate R magic for Google Colab
%load_ext rpy2.ipython

In [ ]:
%%R
# Install required R packages
install.packages(c('RSQLite', 'DBI', 'readr', 'dplyr', 'knitr'), repos='https://cloud.r-project.org')

In [ ]:
%%R
library(RSQLite)
library(DBI)
library(readr)
library(dplyr)


# STEP 1: LOAD CSVs AND CREATE SQLite DATABASE


con <- dbConnect(RSQLite::SQLite(), ":memory:")

# Define zone standardisation function
standardise_zone <- function(z) {
  z <- trimws(z)
  mapping <- c('north'='North','NORTH'='North','south'='South','SOUTH'='South',
               'east'='East','EAST'='East','west'='West','WEST'='West',
               'central'='Central','CENTRAL'='Central','Ctr'='Central',
               'airport'='Airport','AIRPORT'='Airport',
               'riverside'='Riverside','RiverSide'='Riverside')
  ifelse(z %in% names(mapping), mapping[z], z)
}

# Load and clean all datasets

data_path <- '.'

hubs <- read_csv(paste0(data_path, '/hubs.csv'), show_col_types=FALSE)

customers <- read_csv(paste0(data_path, '/customers.csv'), show_col_types=FALSE)
customers$home_zone <- standardise_zone(customers$home_zone)

drivers <- read_csv(paste0(data_path, '/drivers.csv'), show_col_types=FALSE)
drivers$base_zone <- standardise_zone(drivers$base_zone)

vehicles <- read_csv(paste0(data_path, '/vehicles.csv'), show_col_types=FALSE)
vehicles$assigned_zone <- standardise_zone(vehicles$assigned_zone)

orders <- read_csv(paste0(data_path, '/orders.csv'), show_col_types=FALSE)
orders$pickup_zone <- standardise_zone(orders$pickup_zone)
orders$dropoff_zone <- standardise_zone(orders$dropoff_zone)

deliveries <- read_csv(paste0(data_path, '/deliveries.csv'), show_col_types=FALSE)
incidents <- read_csv(paste0(data_path, '/incidents.csv'), show_col_types=FALSE)

complaints <- read_csv(paste0(data_path, '/complaints.csv'), show_col_types=FALSE)

app_events <- read_csv(paste0(data_path, '/app_events.csv'), show_col_types=FALSE)
app_events$zone_context <- standardise_zone(app_events$zone_context)

# Write tables to SQLite
dbWriteTable(con, 'hubs', hubs, overwrite=TRUE)
dbWriteTable(con, 'customers', customers, overwrite=TRUE)
dbWriteTable(con, 'drivers', drivers, overwrite=TRUE)
dbWriteTable(con, 'vehicles', vehicles, overwrite=TRUE)
dbWriteTable(con, 'orders', orders, overwrite=TRUE)
dbWriteTable(con, 'deliveries', deliveries, overwrite=TRUE)
dbWriteTable(con, 'incidents', incidents, overwrite=TRUE)
dbWriteTable(con, 'complaints', complaints, overwrite=TRUE)
dbWriteTable(con, 'app_events', app_events, overwrite=TRUE)

cat('Database loaded successfully.\n')
cat('Tables:', paste(dbListTables(con), collapse=', '), '\n')

In [ ]:
%%R

# QUERY 1: Service Failure Rate by Zone and Hub


q1 <- dbGetQuery(con, "
SELECT 
    o.pickup_zone,
    d.hub_id,
    h.hub_name,
    COUNT(*) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS failure_pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status != 'OnTime' THEN 1 ELSE 0 END) / COUNT(*), 1) AS non_ontime_pct,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost
FROM deliveries d
JOIN orders o ON d.order_id = o.order_id
JOIN hubs h ON d.hub_id = h.hub_id
GROUP BY o.pickup_zone, d.hub_id, h.hub_name
ORDER BY failure_pct DESC
")

cat('=== Query 1: Failure Rate by Zone and Hub ===\n')
print(q1)
cat('\nInterpretation: Central zone (via H05 Central Core and H08 Midtown Relay)\n')
cat('shows the highest failure rates (up to 20.3%), confirming the operations\n')
cat('director\'s concern. H08 (Charging hub) has the worst performance despite\n')
cat('being a support facility, suggesting charging infrastructure bottlenecks.\n')

In [ ]:
%%R

# QUERY 2: Repeat Complainers with Cross-System Status Mismatch


q2 <- dbGetQuery(con, "
SELECT 
    c.customer_id,
    c.home_zone,
    c.loyalty_score,
    COUNT(DISTINCT cp.complaint_id) AS total_complaints,
    COUNT(DISTINCT CASE WHEN d.delivery_status = 'OnTime' THEN cp.complaint_id END) AS complaints_on_ontime_orders,
    COUNT(DISTINCT i.incident_id) AS linked_incidents,
    GROUP_CONCAT(DISTINCT cp.complaint_type) AS complaint_types
FROM customers c
JOIN complaints cp ON c.customer_id = cp.customer_id
LEFT JOIN deliveries d ON cp.order_id = d.order_id
LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
GROUP BY c.customer_id, c.home_zone, c.loyalty_score
HAVING COUNT(DISTINCT cp.complaint_id) >= 2
ORDER BY total_complaints DESC, linked_incidents DESC
LIMIT 20
")

cat('=== Query 2: Repeat Complainers with Status Mismatches ===\n')
print(q2)
cat('\nInterpretation: Multiple customers have complaints against orders marked\n')
cat('OnTime, yet incidents exist. This confirms the cross-system data mismatch\n')
cat('described in the case study - the delivery system records success while\n')
cat('the complaint and incident systems record failure.\n')

In [ ]:
%%R

# QUERY 3: Route Override Analysis by Driver and Zone

q3 <- dbGetQuery(con, "
SELECT 
    dr.driver_id,
    dr.base_zone,
    dr.driver_rating,
    dr.training_score,
    COUNT(*) AS total_deliveries,
    SUM(d.manual_route_override_count) AS total_overrides,
    ROUND(1.0 * SUM(d.manual_route_override_count) / COUNT(*), 2) AS avg_overrides,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS failure_pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS delay_pct
FROM deliveries d
JOIN drivers dr ON d.driver_id = dr.driver_id
GROUP BY dr.driver_id, dr.base_zone, dr.driver_rating, dr.training_score
HAVING COUNT(*) >= 3
ORDER BY avg_overrides DESC
LIMIT 15
")

cat('=== Query 3: Route Override Patterns by Driver ===\n')
print(q3)
cat('\nInterpretation: Drivers with the highest override rates tend to have\n')
cat('higher failure rates, suggesting overrides are associated with poor\n')
cat('outcomes rather than just road condition adaptations. Low training\n')
cat('scores among high-override drivers support a planning/training gap.\n')

In [ ]:
%%R

# QUERY 4: Financial Loss Analysis - Unprofitable Routes


q4 <- dbGetQuery(con, "
SELECT 
    o.pickup_zone,
    o.dropoff_zone,
    o.service_type,
    COUNT(*) AS deliveries,
    ROUND(AVG(o.order_value), 2) AS avg_order_value,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost,
    ROUND(AVG(o.order_value) - AVG(d.fuel_or_charge_cost), 2) AS avg_margin,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS failure_pct,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * AVG(d.fuel_or_charge_cost) AS wasted_cost_on_failures,
    ROUND(AVG(d.route_distance_km), 1) AS avg_distance_km
FROM deliveries d
JOIN orders o ON d.order_id = o.order_id
GROUP BY o.pickup_zone, o.dropoff_zone, o.service_type
HAVING COUNT(*) >= 5
ORDER BY failure_pct DESC, wasted_cost_on_failures DESC
LIMIT 20
")

cat('=== Query 4: Route Profitability and Wasted Costs ===\n')
print(q4)
cat('\nInterpretation: Routes originating from Central zone show the highest\n')
cat('failure rates combined with significant wasted costs on failed deliveries.\n')
cat('While gross margins appear positive, when factoring in repeat delivery\n')
cat('attempts, compensation, and customer churn, several routes become unprofitable.\n')

In [ ]:
%%R

# QUERY 5: Vehicle Maintenance and Incident Correlation


q5 <- dbGetQuery(con, "
SELECT 
    v.vehicle_id,
    v.vehicle_type,
    v.assigned_zone,
    v.battery_health_pct,
    v.maintenance_status,
    v.odometer_km,
    COUNT(DISTINCT d.delivery_id) AS total_deliveries,
    COUNT(DISTINCT i.incident_id) AS total_incidents,
    SUM(CASE WHEN i.incident_type IN ('VehicleFault','BatteryAlert') THEN 1 ELSE 0 END) AS vehicle_related_incidents,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS avg_cost
FROM vehicles v
LEFT JOIN deliveries d ON v.vehicle_id = d.vehicle_id
LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
GROUP BY v.vehicle_id, v.vehicle_type, v.assigned_zone, 
         v.battery_health_pct, v.maintenance_status, v.odometer_km
HAVING total_incidents > 0
ORDER BY vehicle_related_incidents DESC, battery_health_pct ASC
LIMIT 15
")

cat('=== Query 5: Vehicle Health vs Incident Frequency ===\n')
print(q5)
cat('\nInterpretation: Vehicles with maintenance_status = Active but multiple\n')
cat('BatteryAlert/VehicleFault incidents indicate late detection of faults.\n')
cat('This confirms the case study concern that fault events, scheduling\n')
cat('records and route assignments are not analysed together.\n')

In [ ]:
%%R

# QUERY 6: Comprehensive Hub Performance Dashboard


q6 <- dbGetQuery(con, "
SELECT 
    h.hub_id,
    h.hub_name,
    h.zone,
    h.hub_type,
    h.capacity_score,
    COUNT(DISTINCT d.delivery_id) AS total_deliveries,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END) / COUNT(DISTINCT d.delivery_id), 1) AS fail_pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status='Delayed' THEN 1 ELSE 0 END) / COUNT(DISTINCT d.delivery_id), 1) AS delay_pct,
    COUNT(DISTINCT i.incident_id) AS incidents,
    COUNT(DISTINCT cp.complaint_id) AS complaints,
    ROUND(SUM(d.fuel_or_charge_cost), 2) AS total_fuel_cost,
    ROUND(AVG(d.route_distance_km), 1) AS avg_distance,
    ROUND(AVG(d.manual_route_override_count), 2) AS avg_overrides
FROM hubs h
LEFT JOIN deliveries d ON h.hub_id = d.hub_id
LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
LEFT JOIN complaints cp ON d.order_id = cp.order_id
GROUP BY h.hub_id, h.hub_name, h.zone, h.hub_type, h.capacity_score
ORDER BY fail_pct DESC
")

cat('=== Query 6: Comprehensive Hub Performance ===\n')
print(q6)
cat('\nInterpretation: H08 (Midtown Relay - Charging type) has the highest\n')
cat('failure rate at 20.3% with the lowest capacity score (63) and highest\n')
cat('override rate, followed by H05 (Central Core - Control type) at 20%.\n')
cat('These two Central-zone hubs are the primary bottlenecks.\n')

In [ ]:
%%R

# QUERY OPTIMISATION: Indexing and EXPLAIN QUERY PLAN


cat('=== BEFORE INDEXING: Query Plan for Hub Performance Query ===\n')
plan_before <- dbGetQuery(con, "
EXPLAIN QUERY PLAN
SELECT d.hub_id, COUNT(*), AVG(d.fuel_or_charge_cost)
FROM deliveries d
JOIN orders o ON d.order_id = o.order_id
WHERE d.delivery_status = 'Failed'
GROUP BY d.hub_id
")
print(plan_before)

# Create indexes for frequently queried columns
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_deliveries_order_id ON deliveries(order_id)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_deliveries_hub_id ON deliveries(hub_id)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_deliveries_status ON deliveries(delivery_status)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_deliveries_driver ON deliveries(driver_id)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_deliveries_vehicle ON deliveries(vehicle_id)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_orders_pickup_zone ON orders(pickup_zone)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_complaints_order ON complaints(order_id)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_complaints_customer ON complaints(customer_id)')
dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_incidents_delivery ON incidents(delivery_id)')


dbExecute(con, 'CREATE INDEX IF NOT EXISTS idx_del_status_hub ON deliveries(delivery_status, hub_id)')

cat('\n=== AFTER INDEXING: Query Plan for Same Query ===\n')
plan_after <- dbGetQuery(con, "
EXPLAIN QUERY PLAN
SELECT d.hub_id, COUNT(*), AVG(d.fuel_or_charge_cost)
FROM deliveries d
JOIN orders o ON d.order_id = o.order_id
WHERE d.delivery_status = 'Failed'
GROUP BY d.hub_id
")
print(plan_after)

cat('\nOptimisation Justification:\n')
cat('- idx_deliveries_order_id: Speeds up JOINs between deliveries and orders (most common join)\n')
cat('- idx_del_status_hub: Composite index for filtering by status and grouping by hub\n')
cat('- idx_incidents_delivery: Speeds up incident lookups per delivery\n')
cat('- idx_complaints_order: Enables efficient complaint-to-order linking\n')
cat('The EXPLAIN QUERY PLAN output shows the query planner using indexes\n')
cat('for lookups instead of full table scans after indexing.\n')

In [ ]:
%%R

# QUERY 7: Orders Without Deliveries (Data Gap Analysis)


q7 <- dbGetQuery(con, "
SELECT 
    o.service_type,
    o.pickup_zone,
    o.priority_level,
    COUNT(*) AS unmatched_orders,
    ROUND(AVG(o.order_value), 2) AS avg_lost_value
FROM orders o
LEFT JOIN deliveries d ON o.order_id = d.order_id
WHERE d.delivery_id IS NULL
GROUP BY o.service_type, o.pickup_zone, o.priority_level
HAVING COUNT(*) >= 3
ORDER BY unmatched_orders DESC
")

cat('=== Query 7: Orders Without Delivery Records ===\n')
print(head(q7, 20))
cat('\nInterpretation: 300 orders (24%) have no delivery record at all.\n')
cat('This represents a significant data gap - these are either cancelled\n')
cat('orders, system failures, or lost records. The finance director cannot\n')
cat('accurately assess profitability with 24% of order data disconnected.\n')

In [ ]:
%%R

# QUERY 8: Compensation Cost Analysis


q8 <- dbGetQuery(con, "
SELECT 
    o.pickup_zone,
    cp.complaint_type,
    COUNT(*) AS complaint_count,
    ROUND(SUM(cp.compensation_amount), 2) AS total_compensation,
    ROUND(AVG(cp.compensation_amount), 2) AS avg_compensation,
    ROUND(AVG(cp.resolution_days), 1) AS avg_resolution_days,
    SUM(CASE WHEN cp.status IN ('Open','Escalated') THEN 1 ELSE 0 END) AS still_open
FROM complaints cp
JOIN orders o ON cp.order_id = o.order_id
GROUP BY o.pickup_zone, cp.complaint_type
HAVING COUNT(*) >= 3
ORDER BY total_compensation DESC
")

cat('=== Query 8: Compensation Costs by Zone and Complaint Type ===\n')
print(head(q8, 15))

# Disconnect
dbDisconnect(con)
cat('\nDatabase connection closed.\n')